## **Notebook 01 — Config & Connection Check**
- Verify that PostgreSQL + pgvector, Redis, and your API keys are all reachable before writing any RAG code

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from project root

DATABASE_URL  = os.getenv("DATABASE_URL")
DATABASE_SYNC = os.getenv("DATABASE_URL_SYNC")
REDIS_URL     = os.getenv("REDIS_URL")
OPENAI_KEY    = os.getenv("OPENAI_API_KEY")

In [3]:
import psycopg2

# psycopg2 uses the sync URL (postgresql+asyncpg won't work here)
conn_str = DATABASE_SYNC.replace("postgresql+psycopg2://", "postgresql://")

try:
    conn = psycopg2.connect(conn_str)
    cur  = conn.cursor()

    # 1. Check pgvector extension is installed
    cur.execute("SELECT extname, extversion FROM pg_extension WHERE extname='vector';")
    ext = cur.fetchone()
    print("pgvector extension :", ext if ext else "NOT FOUND — rerun Step 1")

    # 2. Check our tables exist
    cur.execute("""
        SELECT table_name FROM information_schema.tables
        WHERE table_schema = 'public'
        ORDER BY table_name;
    """)
    tables = [row[0] for row in cur.fetchall()]
    print("Tables found       :", tables)

    # 3. Check HNSW index exists
    cur.execute("""
        SELECT indexname FROM pg_indexes
        WHERE tablename = 'chunks';
    """)
    indexes = [row[0] for row in cur.fetchall()]
    print("Indexes on chunks  :", indexes)

    conn.close()
    print("\nPostgreSQL         : OK")

except Exception as e:
    print("PostgreSQL FAILED  :", e)

pgvector extension : ('vector', '0.8.2')
Tables found       : []
Indexes on chunks  : ['chunks_pkey', 'idx_chunks_embedding', 'idx_chunks_metadata']

PostgreSQL         : OK


In [4]:
import redis

try:
    client = redis.from_url(REDIS_URL)
    pong   = client.ping()
    print("Redis ping :", "PONG — OK" if pong else "No response")

    # Write and read a test key
    client.set("rag_test_key", "hello", ex=10)  # expires in 10 seconds
    val = client.get("rag_test_key")
    print("Redis read :", val.decode())

except Exception as e:
    print("Redis FAILED :", e)

Redis ping : PONG — OK
Redis read : hello


In [5]:
from langchain_openai import ChatOpenAI

try:
    llm      = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    response = llm.invoke("Reply with the single word: working")
    print("OpenAI response :", response.content)
    print("OpenAI key      : OK")
except Exception as e:
    print("OpenAI FAILED   :", e)

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')


OpenAI response : Working
OpenAI key      : OK


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"API key has expired"}\n')


In [8]:
from sqlalchemy.ext.asyncio import create_async_engine, AsyncSession
from sqlalchemy import text

engine = create_async_engine(DATABASE_URL, echo=False)

async def test_async_db():
    async with AsyncSession(engine) as session:
        result = await session.execute(text("SELECT COUNT(*) FROM chunks;"))
        count  = result.scalar()
        print("Chunks in DB (async) :", count)

# Run in notebook with await
await test_async_db()

Chunks in DB (async) : 0


In [9]:
checks = {
    "PostgreSQL connection" : True,   # update to False if any cell above failed
    "pgvector extension"   : True,
    "chunks table"         : True,
    "HNSW index"           : True,
    "Redis"                : True,
    "OpenAI key"           : True,
    "SQLAlchemy async"     : True,
}

print("=" * 40)
print("INFRASTRUCTURE READINESS REPORT")
print("=" * 40)
for check, status in checks.items():
    icon = "OK" if status else "FAIL"
    print(f"  {icon}   {check}")

all_ok = all(checks.values())
print("=" * 40)
print("READY FOR STEP 3" if all_ok else "FIX FAILURES ABOVE FIRST")

INFRASTRUCTURE READINESS REPORT
  OK   PostgreSQL connection
  OK   pgvector extension
  OK   chunks table
  OK   HNSW index
  OK   Redis
  OK   OpenAI key
  OK   SQLAlchemy async
READY FOR STEP 3
